In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U bitsandbytes accelerate transformers

In [1]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"   # big downloads go here

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    ),
    device_map="auto",
)

vulnerable = '''import sqlite3

def get_user(conn, uid):
    cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE id=" + uid)
    return cur.fetchone()
'''

messages = [
    {"role": "system", "content": "You are a secure coding assistant. Return only the fixed Python code in one code block."},
    {"role": "user", "content": "Fix the security vulnerability in this code:\n\n" + vulnerable},
]
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

```python
import sqlite3

def get_user(conn, uid):
    cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE id=?", (uid,))
    return cur.fetchone()
```


In [2]:
import glob, json, re, sys, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

det = glob.glob("/kaggle/input/**/detector.py", recursive=True)[0]
folder = os.path.dirname(det)
sys.path.append(folder)
from detector import scan

pairs = [json.loads(l) for l in open(folder + "/verified_pairs.jsonl")]
print("Loaded", len(pairs), "verified pairs")


def similar(code, k=2):
    corpus = [p["vulnerable_code"] for p in pairs]
    vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]+")
    m = vec.fit_transform(corpus + [code])
    scores = cosine_similarity(m[-1], m[:-1]).ravel()
    return [pairs[i] for i in scores.argsort()[::-1][:k]]


def ask(messages):
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def extract(text):
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (m.group(1) if m else text).strip()


def safe_scan(code):
    try:
        return scan(code)
    except SyntaxError:
        return [(0, "Fixed code is not valid Python")]


def fix(code, tries=3):
    findings = scan(code)
    if not findings:
        return code, "no vulnerability found"
    examples = ""
    for ex in similar(code):
        examples += f"VULNERABLE:\n{ex['vulnerable_code']}\nFIXED:\n{ex['fixed_code']}\n\n"
    problems = "\n".join(f"line {l}: {msg}" for l, msg in findings)
    prompt = (f"Verified examples of similar fixes:\n\n{examples}"
              f"Now fix this code. The scanner found:\n{problems}\n\n"
              f"Code:\n{code}\n\nReturn the full fixed code in one code block.")
    messages = [
        {"role": "system", "content": "You are a secure coding assistant. Keep behavior the same; remove the vulnerability."},
        {"role": "user", "content": prompt},
    ]
    for attempt in range(1, tries + 1):
        fixed = extract(ask(messages))
        left = safe_scan(fixed)
        if not left:
            return fixed, f"VERIFIED after {attempt} attempt(s)"
        messages.append({"role": "assistant", "content": fixed})
        messages.append({"role": "user", "content": "The scanner still reports:\n" +
                         "\n".join(f"line {l}: {m}" for l, m in left) + "\nFix it and return the full code."})
    return fixed, "NOT verified"


tests = [
    'import os\n\ndef ping(host):\n    os.system("ping -c 1 " + host)\n',
    'def calc(expr):\n    return eval(expr)\n',
    'import sqlite3\n\ndef find(conn, name):\n    cur = conn.cursor()\n    cur.execute(f"SELECT * FROM users WHERE name=\'{name}\'")\n    return cur.fetchall()\n',
]
for t in tests:
    result, status = fix(t)
    print("=" * 60, "\n", status, "\n", result)

Loaded 78 verified pairs
 VERIFIED after 1 attempt(s) 
 import subprocess

def ping(host):
    command = ['ping', '-c', '1', host]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout.strip())
 VERIFIED after 1 attempt(s) 
 import ast
import operator as op

# Supported operators
operators = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
            ast.Div: op.truediv, ast.USub: op.neg}

def evaluate_expr(node):
    if isinstance(node, ast.Num):  # <number>
        return node.n
    elif isinstance(node, ast.BinOp):  # <left> <operator> <right>
        return operators[type(node.op)](evaluate_expr(node.left), evaluate_expr(node.right))
    elif isinstance(node, ast.UnaryOp):  # <operator> <operand>
        return operators[type(node.op)](evaluate_expr(node.operand))
    else:
        raise TypeError(node)

def calc(expr):
    try:
        tree = ast.parse(expr, mode='eval')  # Parse user input
        return evaluate_expr(tree.body)  # Eval

In [3]:
import random, ast, json, time
random.seed(42)

data = pairs[:]
random.shuffle(data)
test, pool = data[:15], data[15:]   # 15 held out for testing


def similar_from(code, pool, k=2):
    corpus = [p["vulnerable_code"] for p in pool]
    vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]+")
    m = vec.fit_transform(corpus + [code])
    scores = cosine_similarity(m[-1], m[:-1]).ravel()
    return [pool[i] for i in scores.argsort()[::-1][:k]]


def func_names(code):
    try:
        return {n.name for n in ast.walk(ast.parse(code))
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))}
    except SyntaxError:
        return set()


def fix_eval(code, use_retrieval, tries=3):
    problems = "\n".join(f"line {l}: {m}" for l, m in scan(code))
    examples = ""
    if use_retrieval:
        for ex in similar_from(code, pool):
            examples += f"VULNERABLE:\n{ex['vulnerable_code']}\nFIXED:\n{ex['fixed_code']}\n\n"
    prompt = ("Verified examples of similar fixes:\n\n" + examples if examples else "") + \
        f"Fix this code. The scanner found:\n{problems}\n\nCode:\n{code}\n\nReturn the full fixed code in one code block."
    messages = [
        {"role": "system", "content": "You are a secure coding assistant. Keep behavior the same; remove the vulnerability."},
        {"role": "user", "content": prompt},
    ]
    for attempt in range(1, tries + 1):
        fixed = extract(ask(messages))
        left = safe_scan(fixed)
        kept = func_names(code) <= func_names(fixed)
        if not left and kept:
            return True, attempt
        feedback = ("The scanner still reports:\n" + "\n".join(f"line {l}: {m}" for l, m in left)
                    if left else "You removed functions. Keep every original function name.")
        messages.append({"role": "assistant", "content": fixed})
        messages.append({"role": "user", "content": feedback + "\nFix it and return the full code."})
    return False, tries


results = {}
for mode in (True, False):
    label = "with retrieval" if mode else "without retrieval"
    ok, attempts = 0, []
    for i, p in enumerate(test, 1):
        t0 = time.time()
        success, n = fix_eval(p["vulnerable_code"], mode)
        ok += success
        attempts.append(n)
        print(f"[{label}] {i}/15 {'OK' if success else 'FAIL'} ({time.time()-t0:.0f}s)")
    results[label] = {"verified": ok, "total": len(test), "avg_attempts": sum(attempts) / len(attempts)}

print("\n===== RESULTS =====")
print(json.dumps(results, indent=2))
json.dump(results, open("/kaggle/working/results.json", "w"), indent=2)

[with retrieval] 1/15 OK (25s)
[with retrieval] 2/15 OK (25s)
[with retrieval] 3/15 OK (13s)
[with retrieval] 4/15 OK (54s)
[with retrieval] 5/15 OK (22s)
[with retrieval] 6/15 OK (26s)
[with retrieval] 7/15 OK (12s)
[with retrieval] 8/15 OK (38s)
[with retrieval] 9/15 OK (17s)
[with retrieval] 10/15 OK (11s)
[with retrieval] 11/15 OK (39s)
[with retrieval] 12/15 OK (17s)
[with retrieval] 13/15 OK (18s)
[with retrieval] 14/15 OK (27s)
[with retrieval] 15/15 OK (20s)
[without retrieval] 1/15 OK (61s)
[without retrieval] 2/15 OK (35s)
[without retrieval] 3/15 OK (25s)
[without retrieval] 4/15 FAIL (73s)
[without retrieval] 5/15 OK (41s)
[without retrieval] 6/15 FAIL (6s)
[without retrieval] 7/15 OK (23s)
[without retrieval] 8/15 OK (26s)
[without retrieval] 9/15 OK (25s)
[without retrieval] 10/15 OK (29s)
[without retrieval] 11/15 OK (29s)
[without retrieval] 12/15 OK (28s)
[without retrieval] 13/15 OK (21s)
[without retrieval] 14/15 OK (40s)
[without retrieval] 15/15 OK (18s)

===== RES